In [134]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os
import warnings
from typing import TypedDict, Literal, Any
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import json
from langgraph.graph import StateGraph, START, END
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from typing import Any
import subprocess
from typing import TypedDict, Literal, Any, Annotated
import operator



pd.set_option('display.max_columns', 100)
warnings.simplefilter(action='ignore', category=FutureWarning)

In [135]:
load_dotenv()

True

In [136]:
API_KEY = os.getenv("OPENROUTER_API_KEY")
print(f"API Key: {API_KEY is not None}")

API Key: True


In [137]:
from typing import TypedDict, Any, Annotated
import operator


class DataDescriptionState(TypedDict, total=False):
    dataset_path: str
    target_column: str | None
    artifacts_dir: str

    basic_overview: dict[str, Any]
    domain_understanding: dict[str, Any]
    schema_detection: dict[str, Any]
    data_quality: dict[str, Any]
    statistics: dict[str, Any]

    final_result: dict[str, Any]
    artifacts: dict[str, str]

    logs: Annotated[list[dict[str, Any]], operator.add]
    error: str | None

## tools

In [138]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from typing import Any
import io
import contextlib
import traceback
from pathlib import Path


class PythonCodeInput(BaseModel):
    code: str = Field(description="Python code to execute.")


@tool(args_schema=PythonCodeInput)
def python_interpreter_tool(code: str) -> dict[str, Any]:
    """
    Executes Python code in isolated local memory.
    Does not return variables or dataframe previews.
    """

    forbidden_patterns = [
        "rm -rf",
        "shutil.rmtree",
        "os.remove",
        "os.rmdir",
        "subprocess",
        "socket",
        "requests.",
        "urllib",
    ]

    lowered_code = code.lower()

    if any(pattern.lower() in lowered_code for pattern in forbidden_patterns):
        return {
            "status": "failed",
            "stdout": "",
            "error": "Code rejected by safety filter.",
        }

    local_memory = {}

    stdout_buffer = io.StringIO()

    try:
        with contextlib.redirect_stdout(stdout_buffer):
            exec(code, local_memory, local_memory)

        return {
            "status": "success",
            "stdout": stdout_buffer.getvalue()[:5000],
            "error": None,
        }

    except Exception:
        return {
            "status": "failed",
            "stdout": stdout_buffer.getvalue()[:5000],
            "error": traceback.format_exc(),
        }

## agent_graph

In [139]:
model = "deepseek/deepseek-v4-flash"
# model = 'google/gemma-4-26b-a4b-it'
model = 'openai/gpt-4.1-mini'
model = 'google/gemini-3-flash-preview'

llm = ChatOpenAI(
    model=model,
    api_key=API_KEY,
    base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
    temperature=0,
    max_tokens=5000
)

In [140]:
from typing import TypedDict, Any, Annotated
import operator


class DataDescriptionState(TypedDict, total=False):
    dataset_path: str
    target_column: str | None
    artifacts_dir: str

    basic_overview: dict[str, Any]
    domain_understanding: dict[str, Any]
    schema_detection: dict[str, Any]
    data_quality: dict[str, Any]
    statistics: dict[str, Any]

    final_result: dict[str, Any]
    artifacts: dict[str, str]

    logs: Annotated[list[dict[str, Any]], operator.add]
    error: str | None

In [141]:
from langchain.agents import create_agent

DATA_DESCRIPTION_STAGE_PROMPT = """
You are a Data Description Agent in a multi-agent ML system.

You analyze datasets step by step.

Rules:
- Use python_interpreter_tool when code execution is needed.
- Write short Python code only for the current stage.
- Do not solve all EDA tasks at once.
- Do not preprocess data.
- Do not train models.
- Do not modify the original dataset.
- Do not use internet or APIs.
- Use only pandas, numpy, json, pathlib, re, collections.
- Return only valid JSON for the current stage.
- Do not include Python code in the final answer.
- Do not include tool calls, tool code, stdout, or debug output in the final answer.
- Do not use markdown.
- Do not add explanations outside JSON.
"""


data_description_stage_agent = create_agent(
    model=llm,
    tools=[python_interpreter_tool],
    system_prompt=DATA_DESCRIPTION_STAGE_PROMPT,
)

In [142]:
def extract_json(text: str) -> dict:
    """
    Достает JSON из ответа модели.
    Работает, даже если модель добавила thought, markdown или текст до/после JSON.
    """
    if text is None:
        raise ValueError("Empty response: text is None")

    text = text.strip()

    if not text:
        raise ValueError("Empty response: text is empty")

    # убираем markdown-обертку
    if text.startswith("```json"):
        text = text.replace("```json", "").replace("```", "").strip()
    elif text.startswith("```"):
        text = text.replace("```", "").strip()

    # пробуем распарсить как есть
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # если модель написала thought перед JSON — берем от первой { до последней }
    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        json_text = text[start:end + 1]
        return json.loads(json_text)

    raise ValueError(f"Could not extract JSON from response. First 500 chars: {text[:500]}")

In [143]:
def run_stage_with_llm(stage_name: str, task: str) -> dict:
    print(f"\n========== START STAGE: {stage_name} ==========")

    try:
        result = data_description_stage_agent.invoke({
            "messages": [
                {
                    "role": "user",
                    "content": task,
                }
            ]
        })

        raw = result["messages"][-1].content.strip()

        print("\nRAW RESPONSE:")
        print(raw)

        parsed = extract_json(raw)

        print(f"========== END STAGE: {stage_name} | status=success ==========")
        return parsed

    except Exception as e:
        print(f"========== END STAGE: {stage_name} | status=failed ==========")
        print(f"ERROR: {e}")
        raise


In [144]:
from pathlib import Path
import json


def dd_basic_overview_node(state: DataDescriptionState) -> DataDescriptionState:
    dataset_path = state["dataset_path"]

    task = f"""
Stage: basic_overview

Read the training dataset from:
{dataset_path}

Use python_interpreter_tool to write and execute short Python code.

Your code must:
1. Read the CSV into a local variable df.
2. Calculate:
   - n_rows
   - n_columns
   - columns
   - dtypes
   - missing_values_total
   - duplicate_rows
3. Do not print Python code or debug output.

Important rules:
- Do not preprocess data.
- Do not modify df.
- Do not print Python code or debug output.
- Do not return the full dataframe.
- Do not keep the full dataframe in memory for next stages.
- Each later stage will read the dataset again if needed.
- Return only compact JSON for this stage.

Return JSON only:
{{
  "stage": "basic_overview",
  "status": "success",
  "result": {{
    "n_rows": 0,
    "n_columns": 0,
    "columns": [],
    "dtypes": {{}},
    "missing_values_total": 0,
    "duplicate_rows": 0
  }}
}}
"""

    try:
        parsed = run_stage_with_llm("basic_overview", task)
        basic_overview = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_description/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        basic_overview_path = artifacts_dir / "basic_overview.json"

        with open(basic_overview_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        return {
            "basic_overview": basic_overview,
            "artifacts": {
                **state.get("artifacts", {}),
                "basic_overview_path": str(basic_overview_path),
            },
            "logs": [{
                "agent": "data_description_agent",
                "stage": "basic_overview",
                "status": "success",
                "summary": "Basic dataset overview completed.",
                "artifacts": {
                    "basic_overview_path": str(basic_overview_path),
                },
            }],
        }

    except Exception as e:
        return {
            "error": str(e),
            "logs": [{
                "agent": "data_description_agent",
                "stage": "basic_overview",
                "status": "failed",
                "summary": "Basic overview failed.",
                "reason": str(e),
            }],
        }

In [145]:
from pathlib import Path
import json


def dd_domain_understanding_node(state: DataDescriptionState) -> DataDescriptionState:
    dataset_path = state["dataset_path"]
    basic = state.get("basic_overview", {})

    task = f"""
Stage: domain_understanding

Read the training dataset from:
{dataset_path}

Previous result:
{json.dumps(basic, ensure_ascii=False, indent=2)}

Use python_interpreter_tool to write and execute short Python code.

Your code must:
1. Read the CSV into a local variable df.
2. Inspect only compact samples:
   - df.head(3)
   - column names
   - sample values from important object/string columns
3. Do not print Python code or debug output.
4. Do not print the full dataframe.
5. Do not keep the full dataframe in memory for next stages.

Infer:
- subject area of the dataset
- what one row represents
- what groups of columns describe

Important rules:
- Do not preprocess data.
- Do not modify df.
- Use column names and sample values to infer the domain.
- Return only compact JSON for this stage.

Return JSON only:
{{
  "stage": "domain_understanding",
  "status": "success",
  "result": {{
    "domain_summary": "...",
    "row_meaning": "...",
    "column_groups_description": {{}}
  }}
}}
"""

    try:
        parsed = run_stage_with_llm("domain_understanding", task)
        domain_understanding = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_description/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        domain_understanding_path = artifacts_dir / "domain_understanding.json"

        with open(domain_understanding_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        return {
            "domain_understanding": domain_understanding,
            "artifacts": {
                **state.get("artifacts", {}),
                "domain_understanding_path": str(domain_understanding_path),
            },
            "logs": [{
                "agent": "data_description_agent",
                "stage": "domain_understanding",
                "status": "success",
                "summary": "Domain understanding completed.",
                "artifacts": {
                    "domain_understanding_path": str(domain_understanding_path),
                },
            }],
        }

    except Exception as e:
        return {
            "error": str(e),
            "logs": [{
                "agent": "data_description_agent",
                "stage": "domain_understanding",
                "status": "failed",
                "summary": "Domain understanding failed.",
                "reason": str(e),
            }],
        }

In [146]:
from pathlib import Path
import json


def dd_schema_detection_node(state: DataDescriptionState) -> DataDescriptionState:
    dataset_path = state["dataset_path"]
    target_column = state.get("target_column")
    basic = state.get("basic_overview", {})
    domain = state.get("domain_understanding", {})

    task = f"""
Stage: schema_detection

Read the training dataset from:
{dataset_path}

Target column from user:
{target_column}

Previous results:
basic_overview:
{json.dumps(basic, ensure_ascii=False, indent=2)}

domain_understanding:
{json.dumps(domain, ensure_ascii=False, indent=2)}

Use python_interpreter_tool to write and execute short Python code.

Your code must:
1. Read the CSV into a local variable df.
2. Inspect:
   - df.dtypes
   - df.nunique()
   - sample values for every object/string column
   - average string length for every object/string column
   - date-like string columns
   - id-like columns
   - boolean-like columns
3. Detect schema groups:
   - numeric
   - categorical
   - text
   - datetime
   - boolean_like
   - id
   - target

Strict rules:
- Do not print Python code or debug output.
- Do not return the full dataframe.
- Do not keep the full dataframe in memory for next stages.
- Do NOT classify numeric columns as datetime.
- Do NOT parse numeric columns as dates.
- Datetime detection applies only to object/string/datetime columns.
- Long free-form string columns should be text, not categorical.
- Columns like name, description, neighborhood_overview, host_about, amenities are text if they contain phrases, descriptions, lists, or natural language.
- ID-like columns such as id, host_id, user_id should be id, not numeric.
- Categorical columns are short string columns with repeated categories.
- If target_column does not exist in df, set target_column to null.
- If target_column is None, infer target only if obvious. Otherwise null.
- Every column should appear in at most one feature group, except target tracking.

Return JSON only:
{{
  "stage": "schema_detection",
  "status": "success",
  "result": {{
    "target_column": null,
    "task_type": "unknown",
    "schema": {{
      "numeric": [],
      "categorical": [],
      "text": [],
      "datetime": [],
      "boolean_like": [],
      "id": [],
      "target": []
    }},
    "schema_notes": []
  }}
}}
"""

    try:
        parsed = run_stage_with_llm("schema_detection", task)
        schema_detection = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_description/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        schema_detection_path = artifacts_dir / "schema_detection.json"

        with open(schema_detection_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        return {
            "schema_detection": schema_detection,
            "artifacts": {
                **state.get("artifacts", {}),
                "schema_detection_path": str(schema_detection_path),
            },
            "logs": [{
                "agent": "data_description_agent",
                "stage": "schema_detection",
                "status": "success",
                "summary": "Schema detection completed.",
                "artifacts": {
                    "schema_detection_path": str(schema_detection_path),
                },
            }],
        }

    except Exception as e:
        return {
            "error": str(e),
            "logs": [{
                "agent": "data_description_agent",
                "stage": "schema_detection",
                "status": "failed",
                "summary": "Schema detection failed.",
                "reason": str(e),
            }],
        }

In [147]:
from pathlib import Path
import json


def dd_data_quality_node(state: DataDescriptionState) -> DataDescriptionState:
    dataset_path = state["dataset_path"]
    schema_detection = state.get("schema_detection", {})

    task = f"""
Stage: data_quality

Read the training dataset from:
{dataset_path}

Schema result:
{json.dumps(schema_detection, ensure_ascii=False, indent=2)}

Use python_interpreter_tool to write and execute short Python code.

Your code must:
1. Read the CSV into a local variable df.
2. Calculate:
   - missing values by column
   - total missing values
   - duplicate rows
   - constant columns
   - high-cardinality columns
   - numeric outliers using IQR only for schema["numeric"]

Important rules:
- Do not preprocess data.
- Do not modify df.
- Do not print Python code or debug output.
- Do not return the full dataframe.
- Do not keep the full dataframe in memory for next stages.
- Use schema["numeric"] from the provided schema result.
- If schema["numeric"] is empty, return an empty numeric_outliers_iqr dictionary.
- Return only compact JSON for this stage.
- The values in final JSON must be copied exactly from variables calculated by python_interpreter_tool. Do not estimate or invent values.

Return JSON only:
{{
  "stage": "data_quality",
  "status": "success",
  "result": {{
    "missing_values": {{}},
    "missing_values_total": 0,
    "duplicate_rows": 0,
    "constant_columns": [],
    "high_cardinality_columns": [],
    "numeric_outliers_iqr": {{}}
  }}
}}
"""

    try:
        parsed = run_stage_with_llm("data_quality", task)
        data_quality = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_description/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        data_quality_path = artifacts_dir / "data_quality.json"

        with open(data_quality_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        return {
            "data_quality": data_quality,
            "artifacts": {
                **state.get("artifacts", {}),
                "data_quality_path": str(data_quality_path),
            },
            "logs": [{
                "agent": "data_description_agent",
                "stage": "data_quality",
                "status": "success",
                "summary": "Data quality analysis completed.",
                "artifacts": {
                    "data_quality_path": str(data_quality_path),
                },
            }],
        }

    except Exception as e:
        return {
            "error": str(e),
            "logs": [{
                "agent": "data_description_agent",
                "stage": "data_quality",
                "status": "failed",
                "summary": "Data quality analysis failed.",
                "reason": str(e),
            }],
        }

In [148]:
from pathlib import Path
import json


def dd_statistics_node(state: DataDescriptionState) -> DataDescriptionState:
    dataset_path = state["dataset_path"]
    schema_detection = state.get("schema_detection", {})

    task = f"""
Stage: statistics

Read the training dataset from:
{dataset_path}

Schema result:
{json.dumps(schema_detection, ensure_ascii=False, indent=2)}

Use python_interpreter_tool to write and execute short Python code.

Your code must:
1. Read the CSV into a local variable df.
2. Use schema and target_column from the provided schema result.
3. Calculate detailed statistics:
   - numeric summary for schema["numeric"]
   - categorical summary for schema["categorical"]
   - text summary for schema["text"]
   - datetime summary for schema["datetime"]
   - target properties if target_column exists

Important rules:
- Do not print Python code or debug output.
- Do not return the full dataframe.
- Do not keep the full dataframe in memory for next stages.
- Do not return full detailed statistics in the final JSON.
- Return only compact metadata for this stage.

Return JSON only:
{{
  "stage": "statistics",
  "status": "success",
  "result": {{
    "numeric_columns_count": 0,
    "categorical_columns_count": 0,
    "text_columns_count": 0,
    "datetime_columns_count": 0,
    "target_column": null,
    "statistics_note": "Detailed statistics were calculated for EDA stage."
  }}
}}
"""

    try:
        parsed = run_stage_with_llm("statistics", task)
        statistics = parsed.get("result", {})

        artifacts_dir = Path("artifacts/data_description/stages")
        artifacts_dir.mkdir(parents=True, exist_ok=True)

        statistics_path = artifacts_dir / "statistics.json"

        with open(statistics_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2)

        return {
            "statistics": {
                **statistics,
                "statistics_path": str(statistics_path),
            },
            "artifacts": {
                **state.get("artifacts", {}),
                "statistics_path": str(statistics_path),
            },
            "logs": [{
                "agent": "data_description_agent",
                "stage": "statistics",
                "status": "success",
                "summary": "Statistics stage completed.",
                "artifacts": {
                    "statistics_path": str(statistics_path),
                },
            }],
        }

    except Exception as e:
        return {
            "error": str(e),
            "logs": [{
                "agent": "data_description_agent",
                "stage": "statistics",
                "status": "failed",
                "summary": "Statistics calculation failed.",
                "reason": str(e),
            }],
        }

In [149]:
from pathlib import Path
import json


def dd_build_final_result_node(state: DataDescriptionState) -> DataDescriptionState:
    if state.get("error"):
        return {
            "logs": [{
                "agent": "data_description_agent",
                "stage": "build_final_result",
                "status": "skipped",
                "summary": "Final result build was skipped because previous stage failed.",
                "reason": state.get("error"),
            }],
        }

    basic = state.get("basic_overview", {})
    domain = state.get("domain_understanding", {})
    schema_detection = state.get("schema_detection", {})
    data_quality = state.get("data_quality", {})
    statistics = state.get("statistics", {})
    previous_artifacts = state.get("artifacts", {})

    required_basic_keys = ["n_rows", "n_columns", "columns", "dtypes"]
    missing_basic_keys = [key for key in required_basic_keys if key not in basic]

    if missing_basic_keys:
        error = f"Cannot build final result. Missing basic_overview keys: {missing_basic_keys}"
        return {
            "error": error,
            "logs": [{
                "agent": "data_description_agent",
                "stage": "build_final_result",
                "status": "failed",
                "summary": "Final result build failed.",
                "reason": error,
            }],
        }

    schema = schema_detection.get("schema", {})

    for key in ["numeric", "categorical", "text", "datetime"]:
        schema.setdefault(key, [])

    for key in ["boolean_like", "id", "target"]:
        schema.setdefault(key, [])

    target_column = schema_detection.get("target_column")
    task_type = schema_detection.get("task_type", "unknown")

    if target_column is None:
        task_type = "unknown"

    artifacts_dir = Path("artifacts/data_description")
    artifacts_dir.mkdir(parents=True, exist_ok=True)

    summary_path = artifacts_dir / "data_description_summary.json"
    eda_artifacts_path = artifacts_dir / "eda_artifacts.json"
    report_path = artifacts_dir / "data_description_report.md"

    statistics_path = statistics.get(
        "statistics_path",
        previous_artifacts.get("statistics_path"),
    )

    artifacts = {
        **previous_artifacts,
        "data_description_summary_path": str(summary_path),
        "eda_artifacts_path": str(eda_artifacts_path),
        "data_description_report_path": str(report_path),
        "statistics_path": statistics_path,
        "eda_stages_dir": "artifacts/data_description/stages",
    }

    data_summary = {
        "n_rows": basic.get("n_rows"),
        "n_columns": basic.get("n_columns"),
        "domain_summary": domain.get("domain_summary"),
        "row_meaning": domain.get("row_meaning"),
        "missing_values_total": data_quality.get("missing_values_total"),
        "duplicate_rows": data_quality.get("duplicate_rows"),
        "statistics_path": statistics_path,
        "eda_artifacts_path": str(eda_artifacts_path),
    }

    final_result = {
        "agent": "data_description_agent",
        "status": "success",
        "skipped": False,
        "summary": (
            f"Dataset contains {basic.get('n_rows')} rows and "
            f"{basic.get('n_columns')} columns. Task type: {task_type}."
        ),
        "decisions": {
            "domain_summary": domain.get("domain_summary"),
            "row_meaning": domain.get("row_meaning"),
            "target_column": target_column,
            "task_type": task_type,
            "schema": schema,
            "data_quality_summary": {
                "missing_values_total": data_quality.get("missing_values_total"),
                "duplicate_rows": data_quality.get("duplicate_rows"),
                "constant_columns_count": len(data_quality.get("constant_columns", [])),
                "high_cardinality_columns_count": len(data_quality.get("high_cardinality_columns", [])),
            },
        },
        "artifacts": artifacts,
        "next_input": {
            "target_column": target_column,
            "task_type": task_type,
            "schema": schema,
            "data_summary": data_summary,
        },
        "reason": None,
    }

    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(final_result, f, ensure_ascii=False, indent=2)

    eda_artifacts = {
        "basic_overview": basic,
        "domain_understanding": domain,
        "schema_detection": schema_detection,
        "data_quality": data_quality,
        "statistics": statistics,
        "stage_artifact_paths": {
            key: value
            for key, value in artifacts.items()
            if key.endswith("_path") or key.endswith("_dir")
        },
    }

    with open(eda_artifacts_path, "w", encoding="utf-8") as f:
        json.dump(eda_artifacts, f, ensure_ascii=False, indent=2)

    report_md = (
        "# Data Description Report\n\n"
        "## Summary\n"
        f"{final_result.get('summary')}\n\n"
        "## Domain\n"
        f"{domain.get('domain_summary')}\n\n"
        "## Row meaning\n"
        f"{domain.get('row_meaning')}\n\n"
        "## Target\n"
        f"- Target column: {target_column}\n"
        f"- Task type: {task_type}\n\n"
        "## Dataset shape\n"
        f"- Rows: {basic.get('n_rows')}\n"
        f"- Columns: {basic.get('n_columns')}\n\n"
        "## Data quality summary\n"
        f"- Total missing values: {data_quality.get('missing_values_total')}\n"
        f"- Duplicate rows: {data_quality.get('duplicate_rows')}\n"
        f"- Constant columns count: {len(data_quality.get('constant_columns', []))}\n"
        f"- High-cardinality columns count: {len(data_quality.get('high_cardinality_columns', []))}\n\n"
        "## Schema\n"
        "```json\n"
        f"{json.dumps(schema, ensure_ascii=False, indent=2)}\n"
        "```\n\n"
        "## Artifacts\n"
        f"- Summary: {summary_path}\n"
        f"- EDA artifacts: {eda_artifacts_path}\n"
        f"- Statistics: {statistics_path}\n"
        f"- Stage artifacts dir: {artifacts.get('eda_stages_dir')}\n"
    )

    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report_md)

    print("\nRAW RESPONSE: build_final_result")
    print(json.dumps(final_result, ensure_ascii=False, indent=2))
    print("END RAW RESPONSE\n")

    return {
        "final_result": final_result,
        "artifacts": artifacts,
        "logs": [{
            "agent": "data_description_agent",
            "stage": "build_final_result",
            "status": "success",
            "summary": "Final compact data description result was built and saved.",
            "artifacts": {
                "data_description_summary_path": str(summary_path),
                "eda_artifacts_path": str(eda_artifacts_path),
                "data_description_report_path": str(report_path),
            },
        }],
    }

In [150]:
def route_after_dd_stage(state: DataDescriptionState) -> str:
    if state.get("error"):
        return "stop"
    return "continue"

In [151]:
from langgraph.graph import StateGraph, START, END


def build_data_description_graph():
    workflow = StateGraph(DataDescriptionState)

    workflow.add_node("basic_overview", dd_basic_overview_node)
    workflow.add_node("domain_understanding", dd_domain_understanding_node)
    workflow.add_node("schema_detection", dd_schema_detection_node)
    workflow.add_node("data_quality", dd_data_quality_node)
    workflow.add_node("statistics", dd_statistics_node)
    workflow.add_node("build_final_result", dd_build_final_result_node)

    workflow.add_edge(START, "basic_overview")

    workflow.add_conditional_edges(
        "basic_overview",
        route_after_dd_stage,
        {"continue": "domain_understanding", "stop": END},
    )

    workflow.add_conditional_edges(
        "domain_understanding",
        route_after_dd_stage,
        {"continue": "schema_detection", "stop": END},
    )

    workflow.add_conditional_edges(
        "schema_detection",
        route_after_dd_stage,
        {"continue": "data_quality", "stop": END},
    )

    workflow.add_conditional_edges(
        "data_quality",
        route_after_dd_stage,
        {"continue": "statistics", "stop": END},
    )

    workflow.add_conditional_edges(
        "statistics",
        route_after_dd_stage,
        {"continue": "build_final_result", "stop": END},
    )

    workflow.add_conditional_edges(
        "build_final_result",
        route_after_dd_stage,
        {"continue": END, "stop": END},
    )

    return workflow.compile()

In [152]:

class AgentState(TypedDict, total=False):
    run_id: str | None

    dataset_path: str | None
    target_column: str | None
    task_type: Literal["classification", "regression", "auto"]

    schema: dict[str, list[str]]
    data_summary: dict[str, Any]
    prep_summary: dict[str, Any]
    text_summary: dict[str, Any]
    modeling_summary: dict[str, Any]

    artifacts: dict[str, str | None]
    logs: Annotated[list[dict[str, Any]], operator.add]

    current_step: str | None
    next_action: str | None
    orchestration_decision: dict[str, Any]
    orchestration_history: Annotated[list[dict[str, Any]], operator.add]

    previous_run: dict[str, Any]
    previous_best_model: dict[str, Any]

    retry_count: int
    max_retries: int
    status: str
    error: str | None

In [153]:
def data_description_agent_node(state: AgentState) -> AgentState:
    dataset_path = state.get("dataset_path")
    target_column = state.get("target_column")

    if not dataset_path:
        log_record = {
            "agent": "data_description_agent",
            "status": "failed",
            "skipped": False,
            "summary": "Dataset path is missing.",
            "decisions": {},
            "artifacts": {},
            "next_input": {},
            "reason": "dataset_path is None",
        }

        return {
            "current_step": "data_description_agent",
            "error": "dataset_path is missing",
            "logs": [log_record],
        }

    try:
        data_description_app = build_data_description_graph()

        dd_state = data_description_app.invoke(
            {
                "dataset_path": dataset_path,
                "target_column": target_column,
                "artifacts_dir": "artifacts/data_description",
                "logs": [],
                "error": None,
                "artifacts": {},
            },
            {"recursion_limit": 20},
        )

        final_result = dd_state.get("final_result", {})

        if dd_state.get("error") or not final_result:
            error = dd_state.get("error") or "Data Description subgraph did not produce final_result."

            log_record = {
                "agent": "data_description_agent",
                "status": "failed",
                "skipped": False,
                "summary": "Data Description subgraph failed.",
                "decisions": {},
                "artifacts": dd_state.get("artifacts", {}),
                "next_input": {},
                "reason": error,
                "subgraph_logs": dd_state.get("logs", []),
            }

            return {
                "current_step": "data_description_agent",
                "error": error,
                "artifacts": {
                    **state.get("artifacts", {}),
                    **dd_state.get("artifacts", {}),
                },
                "logs": [log_record],
            }

        next_input = final_result.get("next_input", {})
        decisions = final_result.get("decisions", {})
        artifacts = final_result.get("artifacts", {})

        schema = next_input.get("schema") or decisions.get("schema", {})
        data_summary = next_input.get("data_summary", {})

        log_record = {
            "agent": "data_description_agent",
            "status": final_result.get("status", "success"),
            "skipped": final_result.get("skipped", False),
            "summary": final_result.get("summary", "Data description completed."),
            "decisions": decisions,
            "artifacts": artifacts,
            "next_input": next_input,
            "reason": final_result.get("reason"),
            "subgraph_logs": dd_state.get("logs", []),
        }

        return {
            "current_step": "data_description_agent",
            "target_column": next_input.get("target_column", target_column),
            "task_type": next_input.get("task_type", state.get("task_type")),
            "schema": schema,
            "data_summary": data_summary,
            "artifacts": {
                **state.get("artifacts", {}),
                **artifacts,
            },
            "logs": [log_record],
            "error": None,
        }

    except Exception as e:
        log_record = {
            "agent": "data_description_agent",
            "status": "failed",
            "skipped": False,
            "summary": "Data Description subgraph failed.",
            "decisions": {},
            "artifacts": {},
            "next_input": {},
            "reason": str(e),
        }

        return {
            "current_step": "data_description_agent",
            "error": str(e),
            "logs": [log_record],
        }

In [154]:
data_description_app = build_data_description_graph()

dd_test_state = data_description_app.invoke(
    {
        "dataset_path": "./rent_predictions/airbnb_train_fe_15000.csv",
        "target_column": "price",
        "artifacts_dir": "artifacts/data_description",
        "logs": [],
        "error": None,
    },
    {"recursion_limit": 20},
)


========== START STAGE: basic_overview ==========

RAW RESPONSE:
{
  "stage": "basic_overview",
  "status": "success",
  "result": {
    "n_rows": 15000,
    "n_columns": 36,
    "columns": [
      "id",
      "description",
      "amenities",
      "property_type",
      "room_type",
      "accommodates",
      "bathrooms",
      "bedrooms",
      "beds",
      "host_since",
      "host_location",
      "host_response_time",
      "host_response_rate",
      "host_acceptance_rate",
      "host_is_superhost",
      "host_listings_count",
      "host_total_listings_count",
      "latitude",
      "longitude",
      "city",
      "has_availability",
      "availability_30",
      "availability_60",
      "availability_90",
      "availability_365",
      "number_of_reviews",
      "number_of_reviews_ltm",
      "number_of_reviews_l30d",
      "first_review",
      "last_review",
      "review_scores_rating",
      "review_scores_cleanliness",
      "review_scores_location",
      "revie

In [155]:
for i, log in enumerate(dd_test_state.get("logs", []), 1):
    print(f"\n--- LOG {i} ---")
    print("agent:", log.get("agent"))
    print("stage:", log.get("stage"))
    print("status:", log.get("status"))
    print("summary:", log.get("summary"))
    print("reason:", log.get("reason"))
    print("artifacts:", log.get("artifacts"))


--- LOG 1 ---
agent: data_description_agent
stage: basic_overview
status: success
summary: Basic dataset overview completed.
reason: None
artifacts: {'basic_overview_path': 'artifacts/data_description/stages/basic_overview.json'}

--- LOG 2 ---
agent: data_description_agent
stage: domain_understanding
status: success
summary: Domain understanding completed.
reason: None
artifacts: {'domain_understanding_path': 'artifacts/data_description/stages/domain_understanding.json'}

--- LOG 3 ---
agent: data_description_agent
stage: schema_detection
status: success
summary: Schema detection completed.
reason: None
artifacts: {'schema_detection_path': 'artifacts/data_description/stages/schema_detection.json'}

--- LOG 4 ---
agent: data_description_agent
stage: data_quality
status: success
summary: Data quality analysis completed.
reason: None
artifacts: {'data_quality_path': 'artifacts/data_description/stages/data_quality.json'}

--- LOG 5 ---
agent: data_description_agent
stage: statistics
stat